In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import functional as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEAT = ROOT / "features/lunit_vits16"

manifest = pd.read_csv(ROOT / "data/manifest.csv", dtype={"patient_id": str})
# first patient from yale cohort
row = manifest[manifest.cohort == "yale"].iloc[0]

with h5py.File(FEAT / "yale" / f"{row.patient_id}.h5") as f:
    # stored as fp16; models expect fp32
    feats = f["features"][:].astype(np.float32)  

x = torch.from_numpy(feats)          

In [ ]:
class ABMIL(nn.Module):
    def __init__(self, in_dim=384, hidden_dim=128, attn_dim=64, dropout=0.25):
        super().__init__()
        # projection layer to reduce embedding dimension from 384 to 128
        self.proj_layer = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), #(384, 128)
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.attn_branch_u = nn.Sequential(
            nn.Linear(hidden_dim, attn_dim),
            nn.Sigmoid()
        )
        self.attn_branch_v = nn.Sequential(
            nn.Linear(hidden_dim, attn_dim),
            nn.Tanh()
        )
        self.attn_w = nn.Linear(attn_dim, 1)
        self.linear_head = nn.Linear(hidden_dim, 1)
        
    def forward(self, X):
        h = self.proj_layer(X)
        sig_side = self.attn_branch_u(h)
        tan_side = self.attn_branch_v(h)
        a_scores = self.attn_w(sig_side * tan_side)
        a = F.softmax(a_scores, dim=0)
        # slide/patient embedding
        z = torch.sum(a * h, dim=0)
        logits = self.linear_head(z)
        return logits, a

In [ ]:
# Functions to track dimensions and tensor stats during the forward pass
def describe(name, t):
    print(f"{name:8s} shape={tuple(t.shape)}  min={t.min():.3f}  max={t.max():.3f}  "
          f"mean={t.mean():.3f}  zeros={(t == 0).float().mean():.1%}")
    
def trace(model, x):
    """Print the output of each top-level layer as the real forward pass runs."""
    handles = [m.register_forward_hook(lambda mod, inp, out, name=name: describe(name, out))
               for name, m in model.named_children()]
    try:
        logit, a = model(x)
    finally:
        for h in handles:
            h.remove()                      
    describe("attn weights", a)
    print("sum of weights:", a.sum().item())
    return logit, a

In [ ]:
# Note: the attention weights are only printed after the forward pass - why they are out of order
trace(ABMIL(), x)

proj_layer shape=(1920, 128)  min=0.000  max=6.626  mean=0.545  zeros=60.9%
attn_branch_u shape=(1920, 64)  min=0.082  max=0.935  mean=0.514  zeros=0.0%
attn_branch_v shape=(1920, 64)  min=-0.995  max=0.980  mean=-0.016  zeros=0.0%
attn_w   shape=(1920, 1)  min=-0.229  max=0.417  mean=0.145  zeros=0.0%
linear_head shape=(1,)  min=-0.552  max=-0.552  mean=-0.552  zeros=0.0%
attn weights shape=(1920, 1)  min=0.000  max=0.001  mean=0.001  zeros=0.0%
sum of weights: 0.9999992251396179


(tensor([-0.5524], grad_fn=<UnsafeViewBackward0>),
 tensor([[0.0004],
         [0.0004],
         [0.0004],
         ...,
         [0.0005],
         [0.0005],
         [0.0006]], grad_fn=<SoftmaxBackward0>))

In [ ]:
# Total model parameters
model = ABMIL()
print(f"Total model parameters: {sum(p.numel() for p in model.parameters())}")

Total model parameters: 65986


All parameters in the model come from the linear FC layers:
- one in the projection layer (384 -> 128)
- two in the gated attention pooling layer (both 128 -> 64)
- one in the classifier head (128 -> 1)

In [ ]:
# In eval mode, model should return exactly the same logits -> deterministic
# In training mode, logits should vary slightly since dropout will zero random set of values
model.eval()
l1, _ = model(x); l2, _ = model(x)

print(f"Eval mode: {l1.item(), l2.item()}")
model.train()
l3, _ = model(x); l4, _ = model(x)
print(f"Training mode: {l3.item(), l4.item()}")  

Eval mode: (0.1814996302127838, 0.1814996302127838)
Training mode: (0.19682642817497253, 0.18158358335494995)


In [ ]:
# Ensuring dev ABMIL model hasn't drifted from final ABMIL model
from her2_pcr_pathology_fm.models import ABMIL as ABMILFinal

dev, final = ABMIL(), ABMILFinal()
final.load_state_dict(dev.state_dict())
dev.eval(); final.eval()
print(torch.allclose(dev(x)[0], final(x)[0]))

True
